## 1. Define the system prompt

In [1]:
system_prompt = """

You are a senior data analyst specialized in geospatial analysis.

You have access to the following tools:

- download_viirs_snpp_peru: Download NASA FIRMS per-country VIIRS-SNPP fires for Peru for a given year

"""

## 2. Load (create) tools

In [2]:
from tools import get_peru_fires, process_peru_fires_complete

In [3]:
result = process_peru_fires_complete(year=2015)

In [4]:
display(result[['NAME_1', 'year', 'n_fires']])

,NAME_1,year,n_fires
0,Amazonas,1970,1256
1,Ancash,1970,1335
2,Apurímac,1970,691
3,Arequipa,1970,534
4,Ayacucho,1970,1366
5,Cajamarca,1970,2301
6,Callao,1970,22
7,Cusco,1970,3513
8,Huancavelica,1970,618
9,Huánuco,1970,8716


## 3. Configure your model

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

## 4. Define response format

In [11]:
from pydantic import BaseModel, Field
from typing import Literal

class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")


## 5. Add memory

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 6. Create the agent

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[get_peru_fires]
)

## 7. Run the agent

In [7]:
query = "what fires happened in peru in the year 2015"

In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content= query)]}
    )

In [ ]:
# Print the entire response in formatted JSON
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='what fires happened in peru in the year 2015', additional_kwargs={}, response_metadata={}, id='259919a3-54a0-432f-bf64-0ad6afb0a52d'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_peru_fires', 'arguments': '{"year": 2015}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--873603d0-c29e-4edd-8cc9-deb7b8c283a8-0', tool_calls=[{'name': 'get_peru_fires', 'args': {'year': 2015}, 'id': '3f7f9dd1-1413-4184-9fa5-4415a7138374', 'type': 'tool_call'}], usage_metadata={'input_tokens': 609, 'output_tokens': 98, 'total_tokens': 707, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 77}}),
              ToolMessage(content='\nFire counts in Peru for 2015:\n\n      NAME_1  year  n_fires\n   SanMartín  1970    11557\n     Ucayali  1970   

In [11]:
print(response['messages'][-1].content)

[{'type': 'text', 'text': "In 2015, there were a total of 69,936 fires in Peru. The distribution of fires by province was as follows:\n\n| Province       | Number of Fires |\n|:---------------|:----------------|\n| San Martín     | 11557           |\n| Ucayali        | 11432           |\n| Huánuco        | 8716            |\n| Loreto         | 6522            |\n| Junín          | 6277            |\n| Madre de Dios  | 3730            |\n| Cusco          | 3513            |\n| Pasco          | 2869            |\n| Cajamarca      | 2301            |\n| La Libertad    | 1949            |\n| Piura          | 1682            |\n| Ayacucho       | 1366            |\n| Ancash         | 1335            |\n| Amazonas       | 1256            |\n| Lambayeque     | 830             |\n| Puno           | 822             |\n| Apurímac       | 691             |\n| Lima           | 622             |\n| Huancavelica   | 618             |\n| Arequipa       | 534             |\n| Moquegua       | 508     

In [13]:
# Display as formatted markdown
from IPython.display import Markdown, display

output = response['messages'][-1].content
# Handle both string and list content types
if isinstance(output, list):
    text = output[0]['text'] if output and 'text' in output[0] else str(output)
else:
    text = output

display(Markdown(text))

In 2015, there were a total of 69,936 fires in Peru. The distribution of fires by province was as follows:

| Province       | Number of Fires |
|:---------------|:----------------|
| San Martín     | 11557           |
| Ucayali        | 11432           |
| Huánuco        | 8716            |
| Loreto         | 6522            |
| Junín          | 6277            |
| Madre de Dios  | 3730            |
| Cusco          | 3513            |
| Pasco          | 2869            |
| Cajamarca      | 2301            |
| La Libertad    | 1949            |
| Piura          | 1682            |
| Ayacucho       | 1366            |
| Ancash         | 1335            |
| Amazonas       | 1256            |
| Lambayeque     | 830             |
| Puno           | 822             |
| Apurímac       | 691             |
| Lima           | 622             |
| Huancavelica   | 618             |
| Arequipa       | 534             |
| Moquegua       | 508             |
| Ica            | 460             |
| Tumbes         | 221             |
| Lima Province  | 55              |
| Tacna          | 48              |
| Callao         | 22              |

Please note that the year column in the detailed breakdown shows '1970', but the data corresponds to the 2015 request.